## Análisis RFM

### 1. Importar archivos csv 'clientes_clean' y 'transacciones_clean'

In [1]:
import pandas as pd

# Cargar los datos originales
clientes = pd.read_csv("clientes_clean.csv")        # customer_id, nombre, email, ciudad, canal, fecha_alta
transacciones = pd.read_csv("transacciones_clean.csv")  # transaccion_id, customer_id, product_id, store_id, cantidad, importe, fecha

# Asegurarse que la columna fecha sea tipo datetime
transacciones['fecha'] = pd.to_datetime(transacciones['fecha'])

### 2. Calcular métricas RFM y crear las columnas: recencia,frecuencia y monetario

In [2]:
# --- Calcular métricas RFM ---
rfm = transacciones.groupby('customer_id').agg({
    'fecha': lambda x: (pd.Timestamp.today() - x.max()).days,  # Recencia en días
    'transaction_id': 'count',                                  # Frecuencia
    'importe': 'sum'                                            # Monetario
}).rename(columns={'fecha':'Recencia','transaction_id':'Frecuencia','importe':'Monetario'})

# Incluir todos los clientes, incluso los que no tienen transacciones
rfm = clientes[['customer_id']].merge(rfm, on='customer_id', how='left')

# Manejar clientes sin transacciones
rfm['Recencia'] = rfm['Recencia'].fillna(999)
rfm['Frecuencia'] = rfm['Frecuencia'].fillna(0)
rfm['Monetario'] = rfm['Monetario'].fillna(0)

In [3]:
rfm.shape


(1000, 4)

In [4]:
rfm

,customer_id,Recencia,Frecuencia,Monetario
0,C0001,340.0,3.0,633.07
1,C0002,180.0,7.0,1279.16
2,C0003,399.0,5.0,1140.96
3,C0004,452.0,7.0,1223.93
4,C0005,136.0,13.0,2873.84
...,...,...,...,...
995,C0996,297.0,5.0,1882.29
996,C0997,658.0,4.0,1028.33
997,C0998,140.0,5.0,883.75
998,C0999,999.0,0.0,0.00


### 3. Cálculo de percentiles para segmentación RFM

In [5]:
# --- Calcular percentiles para segmentación ---
p20_r = rfm['Recencia'].quantile(0.2)
p50_r = rfm['Recencia'].quantile(0.5)
p80_r = rfm['Recencia'].quantile(0.8)

p20_f = rfm['Frecuencia'].quantile(0.2)
p50_f = rfm['Frecuencia'].quantile(0.5)
p80_f = rfm['Frecuencia'].quantile(0.8)

p20_m = rfm['Monetario'].quantile(0.2)
p50_m = rfm['Monetario'].quantile(0.5)
p80_m = rfm['Monetario'].quantile(0.8)

In [6]:


percentiles = pd.DataFrame({
    'Recencia':[p20_r, p50_r, p80_r],
    'Frecuencia':[p20_f, p50_f, p80_f],
    'Monetario':[p20_m, p50_m, p80_m]
}, index=['P20','P50','P80'])

print(percentiles)

     Recencia  Frecuencia  Monetario
P20     134.0         1.0    298.834
P50     238.5         5.0   1198.885
P80     627.2        13.0   2976.232


### 4 . Según los percentiles anteriores, se crea la columna segmento_rfm_adapt

In [7]:
# Función para asignar segmento según percentiles
def segment_rfm_adapt(row):
    if row['Recencia'] == 999:
        return "Sin Transacciones"
    elif row['Recencia'] <= p20_r and row['Frecuencia'] >= p80_f and row['Monetario'] >= p80_m:
        return "Alto Valor"
    elif row['Recencia'] <= p50_r and row['Frecuencia'] >= p50_f and row['Monetario'] >= p50_m:
        return "Medio Valor"
    else:
        return "Bajo Valor"

# Aplicar función a la tabla RFM
rfm['segmento_rfm'] = rfm.apply(segment_rfm_adapt, axis=1)

# Contar clientes por segmento
distribucion_segmentos = rfm['segmento_rfm'].value_counts()
print("Distribución de clientes por segmento RFM:")
print(distribucion_segmentos)

Distribución de clientes por segmento RFM:
segmento_rfm
Bajo Valor           526
Medio Valor          298
Sin Transacciones     97
Alto Valor            79
Name: count, dtype: int64


In [8]:
rfm['segmento_rfm'] = rfm.apply(segment_rfm_adapt, axis=1)

In [9]:
rfm 

,customer_id,Recencia,Frecuencia,Monetario,segmento_rfm
0,C0001,340.0,3.0,633.07,Bajo Valor
1,C0002,180.0,7.0,1279.16,Medio Valor
2,C0003,399.0,5.0,1140.96,Bajo Valor
3,C0004,452.0,7.0,1223.93,Bajo Valor
4,C0005,136.0,13.0,2873.84,Medio Valor
...,...,...,...,...,...
995,C0996,297.0,5.0,1882.29,Bajo Valor
996,C0997,658.0,4.0,1028.33,Bajo Valor
997,C0998,140.0,5.0,883.75,Bajo Valor
998,C0999,999.0,0.0,0.00,Sin Transacciones


### 5. Crear una nueva columna 'churn_riesgo'

In [10]:
# Crear churn_riesgo basado en P50 de Recencia
p50_r = 237.5  # tu percentil P50 de Recencia

rfm['churn_riesgo'] = (rfm['Recencia'] > p50_r).astype(int)

# Revisar resultados
print(rfm[['customer_id', 'Recencia', 'churn_riesgo']].head())

  customer_id  Recencia  churn_riesgo
0       C0001     340.0             1
1       C0002     180.0             0
2       C0003     399.0             1
3       C0004     452.0             1
4       C0005     136.0             0


#### 6. Convertir el tipo de dato d cada columna adaptado al formato de Mysql

In [11]:
# Recuerda: customer_id es string, no lo convertimos a int
# Convertir otros campos a tipos compatibles con MySQL
rfm['Recencia'] = rfm['Recencia'].astype(int)
rfm['Frecuencia'] = rfm['Frecuencia'].astype(int)
rfm['Monetario'] = rfm['Monetario'].round(2)
rfm['segmento_rfm'] = rfm['segmento_rfm'].astype(str)
# Crear churn_riesgo como entero 0/1 compatible con MySQL
rfm['churn_riesgo'] = (rfm['Recencia'] > p50_r).astype(int)

#### 7.Normalizar nombres de columnas a minúscula para consistencia con MySQL y agregar fecha_calculo para trazabilidad de futuras actualizaciones del modelo RFM

In [17]:
rfm.columns = rfm.columns.str.lower()

In [21]:
import datetime

rfm['fecha_calculo'] = datetime.date.today()
rfm.columns = rfm.columns.str.lower()


In [22]:
# Ver tipos de datos de todas las columnas
print(rfm.dtypes)


customer_id       object
recencia           int64
frecuencia         int64
monetario        float64
segmento_rfm      object
churn_riesgo       int64
fecha_calculo     object
dtype: object


### 7. Guardar en formato csv para Mysql

In [23]:

rfm.to_csv("clientes_rfm.csv", index=False)

# 📊 Resumen Analítico RFM con Churn

#### 1. Carga de datos
- CSV cargados: `clientes.csv` y `transacciones.csv`
- Columnas convertidas a tipos correctos (`datetime` para fechas)
- Revisión de nombres de columnas y tipos de datos

---

#### 2. Cálculo de métricas RFM
- Agrupación por `customer_id`:
  - **Recencia:** días desde última compra
  - **Frecuencia:** número de transacciones
  - **Monetario:** suma de importe
- Inclusión de clientes sin compras (`left join`):
  - Recencia = 999, Frecuencia = 0, Monetario = 0

---

#### 3. Cálculo de percentiles
- Percentiles P20, P50, P80 para Recencia, Frecuencia y Monetario
- Base para segmentación según comportamiento real

---

#### 4. Segmentación RFM
- Función `segment_rfm`:
  - **Alto Valor:** Recencia ≤ P20, Frecuencia ≥ P80, Monetario ≥ P80
  - **Medio Valor:** Recencia ≤ P50, Frecuencia ≥ P50, Monetario ≥ P50
  - **Bajo Valor:** resto
  - **Sin Transacciones:** Recencia = 999
- Resultado: columna `segmento_rfm`

---

#### 5. Churn (Riesgo de inactividad)
- Umbral basado en **P50 de Recencia (237 días)**
- Columna `churn_riesgo`:
  - 1 → Recencia > 237 días (en riesgo)
  - 0 → Recencia ≤ 237 días (activo)
- Compatible con MySQL (`TINYINT 0/1`)

---

#### 6. Revisión de resultados
- Distribución de clientes por segmento:

| segmento_rfm       | total |
|------------------|-------|
| Bajo Valor        | 526   |
| Medio Valor       | 298   |
| Sin Transacciones | 97    |
| Alto Valor        | 79    |

- `churn_riesgo` coherente con la distribución de Recencia

---

#### 7.  Preparación para MySQL
| Columna        | Tipo pandas | Tipo MySQL |
|----------------|------------|------------|
| customer_id    | object     | VARCHAR(20) |
| Recencia       | int64      | INT        |
| Frecuencia     | int64      | INT        |
| Monetario      | float64    | DECIMAL(10,2) |
| segmento_rfm   | object     | VARCHAR(20) |
| churn_riesgo   | int64      | TINYINT (0/1) |

- Monetario redondeado a 2 decimales
- Churn listo para filtrar y segmentar en Power BI

---

#### 8. Exportación CSV
- Archivo: `clientes_rfm_churn.csv`
- Columnas: `customer_id`, `Recencia`, `Frecuencia`, `Monetario`, `segmento_rfm`, `churn_riesgo`
- Listo para MySQL y Power BI